# 06 — Property Value Distribution Buckets

**Phase:** Processed Output Aggregation  
**Notebook goal:** Create a compact binned summary of assessed property value percentage changes from the local row-level distribution file generated in Notebook 05.

---

## Purpose

Notebook 05 processed the full City of Vancouver Property Tax Report and wrote two files to `data/processed/`:

- `property_value_change_summary.csv` — a quality-control summary with one row per derived metric. This file is small and tracked in Git.
- `property_value_change_distribution.csv` — a row-level file containing the four derived property value change columns for all 1,552,663 property records. This file is approximately 68.47 MB and is **excluded from Git** due to its size.

This notebook reads only the `percentage_value_change` column from the local distribution file and assigns each record to a percentage change bucket. The result is a small aggregated CSV — `property_value_change_distribution_bins.csv` — that is suitable for version control and downstream visualisation.

> **Important caveat:** BC Assessment assessed values are not MLS sale prices, transaction prices, or market values. They are administrative valuations used as the basis for property tax calculations in British Columbia. Year-over-year changes in assessed value do not directly measure market appreciation or depreciation.

> **Scope:** This notebook does not modify, recreate, or delete the large distribution file. It reads from it once and writes only the small binned summary.

## 2. Imports and Paths

We use `pathlib` for cross-platform path handling and `pandas` for all tabular operations. All paths are derived from `PROJECT_ROOT` so the notebook works regardless of where it is launched from.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT       = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
DISTRIBUTION_PATH  = PROCESSED_DATA_DIR / 'property_value_change_distribution.csv'
BINS_OUTPUT_PATH   = PROCESSED_DATA_DIR / 'property_value_change_distribution_bins.csv'

print(f'PROJECT_ROOT       : .')
print(f'PROCESSED_DATA_DIR : data/processed')
print(f'DISTRIBUTION_PATH  : data/processed/property_value_change_distribution.csv')
print(f'BINS_OUTPUT_PATH   : data/processed/property_value_change_distribution_bins.csv')

PROJECT_ROOT       : .
PROCESSED_DATA_DIR : data/processed
DISTRIBUTION_PATH  : data/processed/property_value_change_distribution.csv
BINS_OUTPUT_PATH   : data/processed/property_value_change_distribution_bins.csv


## 3. Input Validation

Before loading data, we confirm the distribution file exists and report its size. Only the `percentage_value_change` column is loaded — the other three derived columns are not needed for bucketing and loading only the required column keeps memory usage low.

If the assert below fails, run Notebook 05 with `RUN_FULL_PROCESSING = True` to generate the distribution file locally.

In [2]:
assert DISTRIBUTION_PATH.exists(), (
    f'Distribution file not found: {DISTRIBUTION_PATH}\n'
    'Ensure Notebook 05 has been run with RUN_FULL_PROCESSING = True '
    'to generate property_value_change_distribution.csv locally.')

file_size_mb = DISTRIBUTION_PATH.stat().st_size / (1024 ** 2)
print(f'File found    : {DISTRIBUTION_PATH.name}')
print(f'File size     : {file_size_mb:.2f} MB')

COL_PCT_CHANGE = 'percentage_value_change'

df = pd.read_csv(DISTRIBUTION_PATH, usecols=[COL_PCT_CHANGE])

print(f'Rows loaded   : {len(df):,}')
print(f'Column loaded : {list(df.columns)}')
print(f'Dtype         : {df[COL_PCT_CHANGE].dtype}')
print(f'Non-null      : {df[COL_PCT_CHANGE].notna().sum():,}')
print(f'Missing       : {df[COL_PCT_CHANGE].isna().sum():,}')

File found    : property_value_change_distribution.csv
File size     : 68.47 MB
Rows loaded   : 1,552,663
Column loaded : ['percentage_value_change']
Dtype         : float64
Non-null      : 1,508,104
Missing       : 44,559


## 4. Bucket Definitions

Ten buckets cover the full range of `percentage_value_change` values, including a catch-all for records where the column could not be computed.

Boundaries are explicit and unambiguous. `Below -50%` is strictly `< -50`, so a value of exactly -50 falls into `-50% to -20%`. This matches the `extreme_low_count` definition in Notebook 05, where extreme-low records are counted as `percentage_value_change < -50`. `20% to 50%` includes 50 exactly (`<= 50`), and `Above 50%` is strictly `> 50`.

The `Missing / not calculable` label is assigned by the function when `pd.isna(value)` is `True`. Records where the previous-year total was null, zero, or negative could not produce a valid percentage change.

In [3]:
BUCKET_MISSING = 'Missing / not calculable'

BUCKET_ORDER = [
    'Below -50%',
    '-50% to -20%',
    '-20% to -10%',
    '-10% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 20%',
    '20% to 50%',
    'Above 50%',
    BUCKET_MISSING,]

print(f'Total buckets : {len(BUCKET_ORDER)}')
for i, b in enumerate(BUCKET_ORDER, 1):
    print(f'  {i:>2}. {b}')

Total buckets : 10
   1. Below -50%
   2. -50% to -20%
   3. -20% to -10%
   4. -10% to 0%
   5. 0% to 5%
   6. 5% to 10%
   7. 10% to 20%
   8. 20% to 50%
   9. Above 50%
  10. Missing / not calculable


## 5. Binned Summary

`assign_percentage_change_bucket` evaluates each value with explicit `if` conditions and returns the matching bucket label as a string. Using a plain Python function with `.apply()` avoids the boundary ambiguity of `pd.cut` interval defaults and makes every boundary immediately readable in the function body.

The function raises `ValueError` for any non-null value that satisfies no condition — this should never trigger given the `> 50` final guard, but makes silent mis-assignment impossible.

The resulting Series is passed to `value_counts(dropna=False)` and reindexed against `BUCKET_ORDER` to preserve the logical ordering defined in Section 4. `percentage_of_records` is rounded to 4 decimal places.

Four assertions confirm correctness before the output is exported:

- **Boundary check — below:** `Below -50%` bucket count must equal the raw count of records where `percentage_value_change < -50`.
- **Boundary check — above:** `Above 50%` bucket count must equal the raw count of records where `percentage_value_change > 50`.
- **Count check:** the sum of `property_count` must equal the number of rows loaded.
- **Percentage check:** `percentage_of_records` must sum to approximately 100% (within 0.01 percentage points).

In [ ]:
def assign_percentage_change_bucket(value):
    if pd.isna(value):
        return 'Missing / not calculable'
    if value < -50:
        return 'Below -50%'
    if value >= -50 and value < -20:
        return '-50% to -20%'
    if value >= -20 and value < -10:
        return '-20% to -10%'
    if value >= -10 and value < 0:
        return '-10% to 0%'
    if value >= 0 and value < 5:
        return '0% to 5%'
    if value >= 5 and value < 10:
        return '5% to 10%'
    if value >= 10 and value < 20:
        return '10% to 20%'
    if value >= 20 and value <= 50:
        return '20% to 50%'
    if value > 50:
        return 'Above 50%'
    raise ValueError(f'Unexpected percentage_value_change value: {value}')


df['bucket'] = df['percentage_value_change'].apply(assign_percentage_change_bucket)

summary = (
    df['bucket']
    .value_counts(dropna=False)
    .reindex(BUCKET_ORDER, fill_value=0)
    .reset_index())
summary.columns = ['bucket', 'property_count']
summary['percentage_of_records'] = (
    summary['property_count'] / len(df) * 100).round(4)

# Boundary count assertions
below_50_raw    = int((df[COL_PCT_CHANGE] < -50).sum())
above_50_raw    = int((df[COL_PCT_CHANGE] > 50).sum())
below_50_bucket = int(summary.loc[summary['bucket'] == 'Below -50%', 'property_count'].iloc[0])
above_50_bucket = int(summary.loc[summary['bucket'] == 'Above 50%',  'property_count'].iloc[0])

assert below_50_bucket == below_50_raw, (
    f'"Below -50%" bucket ({below_50_bucket:,}) does not match '
    f'raw count where percentage_value_change < -50 ({below_50_raw:,})')
assert above_50_bucket == above_50_raw, (
    f'"Above 50%" bucket ({above_50_bucket:,}) does not match '
    f'raw count where percentage_value_change > 50 ({above_50_raw:,})')

total_count = summary['property_count'].sum()
assert total_count == len(df), (
    f'Count mismatch: sum={total_count:,}, expected={len(df):,}')
pct_sum = summary['percentage_of_records'].sum()
assert abs(pct_sum - 100.0) < 0.01, (
    f'Percentages do not sum to approximately 100%: {pct_sum:.4f}%')

print('Validation passed.')
print(f'Below -50% bucket        : {below_50_bucket:,}  (matches raw count < -50)')
print(f'Above 50%  bucket        : {above_50_bucket:,}  (matches raw count > 50)')
print(f'Total rows accounted for : {total_count:,}')
print(f'Percentage sum           : {pct_sum:.4f}%')
print()
display(summary)

Validation passed.
Below -50% bucket        : 445  (matches raw count < -50)
Above 50%  bucket        : 2,228  (matches raw count > 50)
Total rows accounted for : 1,552,663
Percentage sum           : 100.0000%



,bucket,property_count,percentage_of_records
0,Below -50%,445,0.0287
1,-50% to -20%,9251,0.5958
2,-20% to -10%,113031,7.2798
3,-10% to 0%,534081,34.3977
4,0% to 5%,422327,27.2002
5,5% to 10%,233213,15.0202
6,10% to 20%,166091,10.6972
7,20% to 50%,27437,1.7671
8,Above 50%,2228,0.1435
9,Missing / not calculable,44559,2.8698


## 6. Export

The binned summary is written to `data/processed/property_value_change_distribution_bins.csv`. This file is small (10 rows, 3 columns) and is suitable for version control.

In [5]:
summary.to_csv(BINS_OUTPUT_PATH, index=False)

print(f'Written : {BINS_OUTPUT_PATH.name}')
print(f'Path    : data/processed/{BINS_OUTPUT_PATH.name}')
print(f'Rows    : {len(summary):,}')

Written : property_value_change_distribution_bins.csv
Path    : data/processed/property_value_change_distribution_bins.csv
Rows    : 10


## 7. Output Validation

Six checks confirm that the output file is correct and complete before this notebook is considered done. These checks are designed to be re-runnable: if the notebook is re-executed, they confirm that the re-generated output is still valid.

> **Validation note:** `Below -50%` is defined as `percentage_value_change < -50` (strictly less than -50). This matches the `extreme_low_count` definition in Notebook 05, where `EXTREME_LOW_THRESHOLD = -50.0` and extreme-low records are counted as `percentage_value_change < EXTREME_LOW_THRESHOLD`. A value of exactly -50 falls into `-50% to -20%`, not `Below -50%`.

In [6]:
assert BINS_OUTPUT_PATH.exists(), f'Output file not found: {BINS_OUTPUT_PATH}'
print('Output file exists              : OK')

reload = pd.read_csv(BINS_OUTPUT_PATH)

assert len(reload) > 0, 'Output file is empty'
print(f'Output file is non-empty        : {len(reload):,} rows — OK')

EXPECTED_COLS = {'bucket', 'property_count', 'percentage_of_records'}
actual_cols   = set(reload.columns.tolist())
assert EXPECTED_COLS == actual_cols, (
    f'Column mismatch.\n  Expected : {sorted(EXPECTED_COLS)}\n  Got      : {sorted(actual_cols)}')
print('Expected columns present        : OK')

count_sum = reload['property_count'].sum()
assert count_sum == len(df), (
    f'Count sum mismatch: {count_sum:,} != {len(df):,}')
print(f'Property counts sum correctly   : {count_sum:,} — OK')

assert reload['bucket'].nunique() == len(reload), 'Duplicate bucket names found'
print('No duplicate bucket names       : OK')

expected_buckets = set(BUCKET_ORDER)
actual_buckets   = set(reload['bucket'].tolist())
assert expected_buckets == actual_buckets, (
    f'Bucket name mismatch.\n'
    f'  Missing from output : {expected_buckets - actual_buckets}\n'
    f'  Unexpected in output: {actual_buckets - expected_buckets}')
print('All expected buckets present    : OK')

pct_sum_reload = reload['percentage_of_records'].sum()
assert abs(pct_sum_reload - 100.0) < 0.01, (
    f'Reloaded percentages do not sum to approximately 100%: {pct_sum_reload:.4f}%')
print(f'Percentages sum to ~100%        : {pct_sum_reload:.4f}% — OK')

print()
print('Boundary note: "Below -50%" counts rows where percentage_value_change < -50')
print('(strictly less than -50). A value of exactly -50 falls into "-50% to -20%".')
print('This is consistent with extreme_low_count in Notebook 05.')
print()
print('All output validation checks passed.')

Output file exists              : OK
Output file is non-empty        : 10 rows — OK
Expected columns present        : OK
Property counts sum correctly   : 1,552,663 — OK
No duplicate bucket names       : OK
All expected buckets present    : OK
Percentages sum to ~100%        : 100.0000% — OK

Boundary note: "Below -50%" counts rows where percentage_value_change < -50
(strictly less than -50). A value of exactly -50 falls into "-50% to -20%".
This is consistent with extreme_low_count in Notebook 05.

All output validation checks passed.


## 8. Final Status

This cell prints a clear confirmation of what this notebook does, what it produced, and what it does not do.

In [7]:
print('=' * 70)
print('NOTEBOOK STATUS')
print('=' * 70)
print()
print('Input')
print(f'  File : {DISTRIBUTION_PATH.name}')
print(f'  Note : local file, excluded from Git due to size (~68 MB)')
print(f'  Rows : {len(df):,}')
print()
print('Output')
print(f'  File : {BINS_OUTPUT_PATH.name}')
print(f'  Note : small aggregated file, suitable for version control')
print(f'  Rows : {len(summary):,} (one row per bucket)')
print()
print('This notebook does not make causal claims about housing supply,')
print('permitting activity, or market prices. Assessed values are')
print('administrative property valuations, not MLS sale prices.')
print('=' * 70)

NOTEBOOK STATUS

Input
  File : property_value_change_distribution.csv
  Note : local file, excluded from Git due to size (~68 MB)
  Rows : 1,552,663

Output
  File : property_value_change_distribution_bins.csv
  Note : small aggregated file, suitable for version control
  Rows : 10 (one row per bucket)

This notebook does not make causal claims about housing supply,
permitting activity, or market prices. Assessed values are
administrative property valuations, not MLS sale prices.
